In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
import mlflow.sklearn
import os
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# ── DAGSHUB CONNECTION ────────────────────────────────────
os.environ["MLFLOW_TRACKING_USERNAME"] = "kende23"
os.environ["MLFLOW_TRACKING_PASSWORD"] = "be7a74e194c6b7c0d666a4938cfee49142c7d603"
mlflow.set_tracking_uri("https://dagshub.com/kende23/ML-homework-1.mlflow")

# ── CREATE OR GET EXPERIMENT ─────────────────────────────
experiment_name = "house-prices"

experiment = mlflow.get_experiment_by_name(experiment_name)
if experiment is None:
    mlflow.create_experiment(experiment_name)
    print(f"Created experiment: {experiment_name}")
else:
    print(f"Found existing experiment: {experiment_name}")

mlflow.set_experiment(experiment_name)
print("MLflow connected!")

Found existing experiment: house-prices
MLflow connected!


In [5]:
# ── LOAD DATA ─────────────────────────────────────────────
import pandas as pd
import numpy as np

X_train = pd.read_csv("../data/X_train_all.csv")
X_val   = pd.read_csv("../data/X_val_all.csv")
X_test  = pd.read_csv("../data/X_test_all.csv")

# you need y too — reload from original
train = pd.read_csv("../house-prices-advanced-regression-techniques/train.csv")
y = np.log1p(train["SalePrice"])

# split y same way you split X (80/20)
from sklearn.model_selection import train_test_split
_, _, y_train, y_val = train_test_split(
    train.drop(columns=["SalePrice"]), y,
    test_size=0.2, random_state=42
)

print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_val:",   X_val.shape,   "y_val:",   y_val.shape)

X_train: (1155, 216) y_train: (1168,)
X_val: (289, 216) y_val: (292,)


In [6]:
# ── DEFINE MODELS ─────────────────────────────────────────
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor

models = {
    "linear_regression": LinearRegression(),
    "decision_tree":     DecisionTreeRegressor(random_state=42),
}

In [9]:
# ── TRAIN + LOG ───────────────────────────────────────────
results = {}

for name, model in models.items():
    with mlflow.start_run(run_name=name):

        model.fit(X_train, y_train)
        val_preds = model.predict(X_val)

        rmse = np.sqrt(mean_squared_error(y_val, val_preds))
        mae  = mean_absolute_error(y_val, val_preds)
        r2   = r2_score(y_val, val_preds)

        mlflow.log_params(model.get_params())
        mlflow.log_metric("val_rmse", rmse)
        mlflow.log_metric("val_mae",  mae)
        mlflow.log_metric("val_r2",   r2)
        mlflow.sklearn.log_model(model, name)

        results[name] = {"rmse": rmse, "mae": mae, "r2": r2}
        print(f"{name:25s}  RMSE={rmse:.4f}  MAE={mae:.4f}  R²={r2:.4f}")

print("\nDone!")

🏃 View run linear_regression at: https://dagshub.com/kende23/ML-homework-1.mlflow/#/experiments/1/runs/5f0e5a2f6cc34f3995a2e978e28af623
🧪 View experiment at: https://dagshub.com/kende23/ML-homework-1.mlflow/#/experiments/1


ValueError: Found input variables with inconsistent numbers of samples: [1155, 1168]